# Chain-of-Thought vs Standard Prompting on GSM8K

**Author:** [Your Name]  
**Blog:** [Your Hashnode Blog Link]  
**Description:** This notebook verifies the CoT hypothesis from Wei et al. (2022) using Qwen 2.5 1.5B on 10 GSM8K test problems.

## Research Question
Does adding 'Let's think step by step' improve accuracy on math reasoning tasks, even on a small 1.5B parameter model?

## Setup
- **Model:** Qwen 2.5 1.5B Instruct
- **Dataset:** GSM8K (grade school math)
- **Comparison:** Standard prompting vs. Zero-Shot CoT
- **Sample:** 10 test problems

## Step 1: Install Dependencies

In [ ]:
!pip install datasets transformers accelerate

## Step 2: Load GSM8K Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('gsm8k', 'main')
print(dataset)

# Preview first 3 problems
for i in range(3):
    print(f'\n--- Problem {i+1} ---')
    print('Question:', dataset['test'][i]['question'])
    print('Answer:', dataset['test'][i]['answer'])

## Step 3: Extract Final Answers

In [ ]:
def extract_answer(answer_text):
    """Extract final answer after #### marker in GSM8K format."""
    return answer_text.split('####')[-1].strip()

# Test extraction
for i in range(3):
    answer = extract_answer(dataset['test'][i]['answer'])
    print(f'Problem {i+1} Final Answer: {answer}')

## Step 4: Load Qwen 2.5 1.5B Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

print('Model loaded successfully!')

## Step 5: Define Inference Functions

In [ ]:
def solve_without_cot(question):
    """Solve math problem using standard prompting."""
    prompt = f'Answer this math question with just the final number:\n{question}'
    inputs = tokenizer(prompt, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()


def solve_with_cot(question):
    """Solve math problem using Zero-Shot Chain-of-Thought prompting."""
    prompt = f'Answer this math question. Think step by step:\n{question}\nLet\'s think step by step:'
    inputs = tokenizer(prompt, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()


def extract_final_number(text):
    """Extract the last number mentioned in model output."""
    import re
    numbers = re.findall(r'\$?(\d+(?:\.\d+)?)', text)
    return numbers[-1] if numbers else None

## Step 6: Run Experiment on 10 Problems

In [ ]:
correct_without_cot = 0
correct_with_cot = 0
total = 10

results = []

for i in range(total):
    question = dataset['test'][i]['question']
    correct = extract_answer(dataset['test'][i]['answer'])
    
    ans_without = extract_final_number(solve_without_cot(question))
    ans_with = extract_final_number(solve_with_cot(question))
    
    no_cot_correct = ans_without == correct
    cot_correct = ans_with == correct
    
    if no_cot_correct:
        correct_without_cot += 1
    if cot_correct:
        correct_with_cot += 1
    
    results.append({
        'problem': i+1,
        'correct': correct,
        'no_cot': ans_without,
        'cot': ans_with,
        'no_cot_correct': no_cot_correct,
        'cot_correct': cot_correct
    })
    
    print(f'Problem {i+1}: Correct={correct} | No CoT={ans_without} | CoT={ans_with}')

print(f'\n--- FINAL RESULTS ---')
print(f'Without CoT: {correct_without_cot}/{total} ({correct_without_cot*10}%)')
print(f'With CoT:    {correct_with_cot}/{total} ({correct_with_cot*10}%)')

## Step 7: Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

no_cot_correct_list = [1 if r['no_cot_correct'] else 0 for r in results]
cot_correct_list = [1 if r['cot_correct'] else 0 for r in results]
problems = [f"P{r['problem']}" for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('CoT vs No-CoT on GSM8K — Qwen 2.5 1.5B', fontsize=14, fontweight='bold')

# Per Problem Results
x = np.arange(10)
width = 0.35

axes[0].bar(x - width/2, no_cot_correct_list, width, label='Without CoT', color='#e74c3c', alpha=0.85)
axes[0].bar(x + width/2, cot_correct_list, width, label='With CoT', color='#2ecc71', alpha=0.85)
axes[0].set_xlabel('Problems')
axes[0].set_ylabel('Correct (1) / Wrong (0)')
axes[0].set_title('Per Problem: Correct vs Wrong')
axes[0].set_xticks(x)
axes[0].set_xticklabels(problems)
axes[0].legend()
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Wrong', 'Correct'])
axes[0].set_ylim(0, 1.3)

for i, (nc, c) in enumerate(zip(no_cot_correct_list, cot_correct_list)):
    if nc == 1:
        axes[0].text(i - width/2, 1.05, '✓', ha='center', color='#c0392b', fontsize=10)
    if c == 1:
        axes[0].text(i + width/2, 1.05, '✓', ha='center', color='#27ae60', fontsize=10)

# Overall Accuracy
accuracies = [correct_without_cot * 10, correct_with_cot * 10]
bars = axes[1].bar(['Without CoT', 'With CoT'], accuracies, color=['#e74c3c', '#2ecc71'], width=0.4)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Overall Accuracy Comparison')
axes[1].set_ylim(0, 100)

for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc}%', ha='center', fontweight='bold', fontsize=12)

axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig('cot_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved as cot_results.png')